# PR4: CNN from Scratch — CIFAR-10 Image Classifier
Notebook: `PR4.ipynb`

Covers A1–A6: CIFAR-10 EDA and preprocessing, convolution concepts, pooling, LeNet-5, custom CNN, callbacks and evaluation, two augmentation methods, MobileNetV2 feature extraction and fine-tuning, results comparison, and a residual block.

**Run cells from top to bottom. Training all models may take a long time on CPU. The test set is reserved for final evaluation.**

## A1 — Load, explore, visualize, normalize, and encode CIFAR-10

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

In [ ]:
# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Load CIFAR-10
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.cifar10.load_data()

In [ ]:
# Define class names
class_names = ["Airplane", "Automobile", "Bird", "Cat", "Deer", "Dog", "Frog", "Horse", "Ship", "Truck"]

In [ ]:
# Print dataset information
print("Train:", X_train_raw.shape, y_train_raw.shape)
print("Test:", X_test_raw.shape, y_test_raw.shape)

In [ ]:
# Show class distribution
counts = pd.Series(y_train_raw.ravel()).value_counts().sort_index()
display(pd.DataFrame({"Class": class_names, "Training Samples": counts.values}))
plt.figure(figsize=(10,4))
plt.bar(class_names, counts.values)
plt.title("CIFAR-10 Training Class Distribution")
plt.xlabel("Class")
plt.ylabel("Images")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 5 images for each of the 10 classes
fig, axes = plt.subplots(5, 10, figsize=(16, 9))
for class_id in range(10):
    indices = np.where(y_train_raw.ravel() == class_id)[0][:5]
    for row, image_index in enumerate(indices):
        axes[row, class_id].imshow(X_train_raw[image_index])
        axes[row, class_id].axis("off")
        if row == 0:
            axes[row, class_id].set_title(class_names[class_id], fontsize=8)
plt.suptitle("CIFAR-10 — 5 Samples per Class")
plt.tight_layout()
plt.show()

In [ ]:
# Normalize pixels from 0–255 to 0–1
X_train = X_train_raw.astype("float32") / 255.0
X_test = X_test_raw.astype("float32") / 255.0

In [ ]:
# One-hot encode labels
y_train_ohe = keras.utils.to_categorical(y_train_raw, 10)
y_test_ohe = keras.utils.to_categorical(y_test_raw, 10)
print("Normalized range:", X_train.min(), "to", X_train.max())
print("One-hot shape:", y_train_ohe.shape)

In [ ]:
# Compare original and normalized pixel distributions
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(X_train_raw[:100].ravel(), bins=30)
ax[0].set_title("Original pixels (0–255)")
ax[1].hist(X_train[:100].ravel(), bins=30)
ax[1].set_title("Normalized pixels (0–1)")
for a in ax:
    a.set_xlabel("Pixel value")
    a.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

**A1 — CNN versus ANN:** A plain ANN commonly flattens an image into a vector, losing explicit spatial structure. CNN filters examine local regions and share weights across the image, making them well suited to image data. Cats/dogs and automobiles/trucks can be visually similar at 32×32 resolution.

## A2 — Convolution, padding, strides, and 16 feature maps

In [ ]:
# Create a small image and a simple edge filter
small_image = np.array([[1,2,0,1,3],[0,1,2,2,1],[3,1,0,1,2],[2,2,1,0,1],[1,0,2,3,1]], dtype="float32")
edge_filter = np.array([[1,0,-1],[1,0,-1],[1,0,-1]], dtype="float32")
image_tensor = tf.constant(small_image.reshape(1,5,5,1))
filter_tensor = tf.constant(edge_filter.reshape(3,3,1,1))
manual_result = tf.nn.conv2d(image_tensor, filter_tensor, strides=1, padding="VALID")
print("Input image:\n", small_image)
print("Filter:\n", edge_filter)
print("Convolution result:\n", manual_result.numpy()[0,:,:,0])

In [ ]:
# Compare padding and strides
demo = tf.constant(np.arange(1,26,dtype="float32").reshape(1,5,5,1))
kernel = tf.ones((3,3,1,1))
print("VALID stride 1:", tf.nn.conv2d(demo,kernel,strides=1,padding="VALID").shape)
print("SAME stride 1:", tf.nn.conv2d(demo,kernel,strides=1,padding="SAME").shape)
print("SAME stride 2:", tf.nn.conv2d(demo,kernel,strides=2,padding="SAME").shape)

In [ ]:
# Extract and display 16 feature maps from a convolution layer
feature_input = keras.Input(shape=(32,32,3))
feature_output = layers.Conv2D(16,3,padding="same",activation="relu")(feature_input)
feature_model = keras.Model(feature_input,feature_output)
maps = feature_model.predict(X_train[:1],verbose=0)
fig, axes = plt.subplots(4,4,figsize=(8,8))
for i, a in enumerate(axes.flat):
    a.imshow(maps[0,:,:,i], cmap="viridis")
    a.set_title(f"Feature map {i+1}", fontsize=8)
    a.axis("off")
plt.suptitle("16 Feature Maps")
plt.tight_layout()
plt.show()

## A3 — Pooling, Flatten, LeNet-5, and custom CNN (<2M parameters)

In [ ]:
# Compare max pooling and average pooling
pool_input = keras.Input(shape=(32,32,3))
max_model = keras.Model(pool_input, layers.MaxPooling2D(2)(pool_input))
avg_model = keras.Model(pool_input, layers.AveragePooling2D(2)(pool_input))
max_img = max_model.predict(X_train[:1],verbose=0)
avg_img = avg_model.predict(X_train[:1],verbose=0)
fig, ax = plt.subplots(1,3,figsize=(9,3))
ax[0].imshow(X_train[0]); ax[0].set_title("Original")
ax[1].imshow(max_img[0,:,:,0],cmap="gray"); ax[1].set_title("Max pool, channel 1")
ax[2].imshow(avg_img[0,:,:,0],cmap="gray"); ax[2].set_title("Average pool, channel 1")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Demonstrate Flatten output shape
flatten_demo = keras.Sequential([keras.Input(shape=(8,8,16)), layers.Flatten()])
print("Flatten output shape:", flatten_demo.output_shape)

In [ ]:
# Build a LeNet-5-style model adapted for RGB and 10 classes
lenet5 = keras.Sequential([
    keras.Input(shape=(32,32,3)),
    layers.Conv2D(6,5,activation="tanh"),
    layers.AveragePooling2D(2,2),
    layers.Conv2D(16,5,activation="tanh"),
    layers.AveragePooling2D(2,2),
    layers.Flatten(),
    layers.Dense(120,activation="tanh"),
    layers.Dense(84,activation="tanh"),
    layers.Dense(10,activation="softmax")
], name="lenet5")
lenet5.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])
lenet5.summary()

In [ ]:
# Train LeNet-5; reduce epochs temporarily for a quick test if needed
es_lenet = keras.callbacks.EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)
history_lenet = lenet5.fit(X_train,y_train_ohe,epochs=20,batch_size=64,validation_split=0.1,callbacks=[es_lenet])
test_loss_lenet,test_accuracy_lenet = lenet5.evaluate(X_test,y_test_ohe,verbose=0)
print("LeNet-5 test accuracy:",test_accuracy_lenet)

In [ ]:
# Create the custom CNN used for tasks A4 and A5
def build_custom_cnn(name="custom_cnn"):
    return keras.Sequential([
        keras.Input(shape=(32,32,3)),
        layers.Conv2D(32,3,padding="same",activation="relu"),
        layers.Conv2D(32,3,padding="same",activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64,3,padding="same",activation="relu"),
        layers.Conv2D(64,3,padding="same",activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(128,3,padding="same",activation="relu"),
        layers.GlobalAveragePooling2D(),
        layers.Dense(256,activation="relu"),
        layers.Dense(10,activation="softmax")
    ],name=name)

model_cnn = build_custom_cnn("cnn_scratch_no_aug")
model_cnn.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])
model_cnn.summary()
print("CNN total parameters:",model_cnn.count_params())
assert model_cnn.count_params() < 2_000_000

## A4 — Train scratch CNN, callbacks, curves, confusion matrix, misclassifications

In [ ]:
# Early stopping and learning-rate reduction
es = keras.callbacks.EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True,verbose=1)
rlrop = keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=5,min_lr=1e-6,verbose=1)

In [ ]:
# Train up to 100 epochs, batch size 64, validation split 10%
history_cnn = model_cnn.fit(
    X_train,y_train_ohe,epochs=100,batch_size=64,validation_split=0.1,
    callbacks=[es,rlrop],verbose=1
)

In [ ]:
# Evaluate on held-out test data
test_loss_cnn,test_accuracy_cnn = model_cnn.evaluate(X_test,y_test_ohe,verbose=0)
print("Scratch CNN test accuracy:",test_accuracy_cnn)

In [ ]:
# Plot training/validation accuracy and loss
fig, ax = plt.subplots(1,2,figsize=(12,4))
ax[0].plot(history_cnn.history["accuracy"],label="Train")
ax[0].plot(history_cnn.history["val_accuracy"],label="Validation")
ax[0].set_title("Scratch CNN Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Accuracy"); ax[0].legend()
ax[1].plot(history_cnn.history["loss"],label="Train")
ax[1].plot(history_cnn.history["val_loss"],label="Validation")
ax[1].set_title("Scratch CNN Loss"); ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Loss"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Predict test classes and print precision/recall/F1 report
pred_prob_cnn = model_cnn.predict(X_test,batch_size=128,verbose=0)
y_pred_cnn = np.argmax(pred_prob_cnn,axis=1)
y_true = y_test_raw.ravel()
print(classification_report(y_true,y_pred_cnn,target_names=class_names,digits=4))

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(y_true,y_pred_cnn)
fig, ax = plt.subplots(figsize=(9,9))
ConfusionMatrixDisplay(cm,display_labels=class_names).plot(ax=ax,xticks_rotation=45,colorbar=False)
plt.title("Scratch CNN — Test Confusion Matrix")
plt.tight_layout(); plt.show()

In [ ]:
# Display up to 15 incorrect predictions
wrong = np.where(y_pred_cnn != y_true)[0]
print("Misclassified images:",len(wrong))
fig, ax = plt.subplots(3,5,figsize=(12,8))
for i,a in enumerate(ax.flat):
    if i < min(15,len(wrong)):
        j=wrong[i]; a.imshow(X_test[j])
        a.set_title(f"True: {class_names[y_true[j]]}\nPred: {class_names[y_pred_cnn[j]]}",fontsize=8)
    a.axis("off")
plt.suptitle("Misclassified Test Images"); plt.tight_layout(); plt.show()

## A5 — ImageDataGenerator and Keras augmentation layers

In [ ]:
# Split 10% of training data for validation; keep test data separate
validation_size = int(len(X_train)*0.1)
X_train_aug, y_train_aug = X_train[:-validation_size], y_train_ohe[:-validation_size]
X_val_aug, y_val_aug = X_train[-validation_size:], y_train_ohe[-validation_size:]

In [ ]:
# Configure ImageDataGenerator augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator
datagen = ImageDataGenerator(rotation_range=15,width_shift_range=0.1,height_shift_range=0.1,
                             horizontal_flip=True,zoom_range=0.1,fill_mode="nearest")
train_generator_aug = datagen.flow(X_train_aug,y_train_aug,batch_size=64,shuffle=True)

In [ ]:
# Display augmented examples from one image
aug_batch, _ = next(datagen.flow(X_train_aug[:1],y_train_aug[:1],batch_size=6,shuffle=False))
fig, ax = plt.subplots(1,6,figsize=(12,3))
for i,a in enumerate(ax):
    a.imshow(aug_batch[i]); a.set_title(f"Aug {i+1}",fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Train custom CNN with ImageDataGenerator
model_aug = build_custom_cnn("cnn_image_data_generator")
model_aug.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])
es_aug = keras.callbacks.EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True,verbose=1)
rlrop_aug = keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=5,min_lr=1e-6,verbose=1)
history_aug = model_aug.fit(train_generator_aug,epochs=100,validation_data=(X_val_aug,y_val_aug),
                            callbacks=[es_aug,rlrop_aug],verbose=1)
test_loss_aug,test_accuracy_aug = model_aug.evaluate(X_test,y_test_ohe,verbose=0)
print("ImageDataGenerator test accuracy:",test_accuracy_aug)

In [ ]:
# Build modern Keras augmentation layers and Functional API CNN
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),layers.RandomTranslation(0.1,0.1)
],name="data_augmentation")
inputs_aug = keras.Input(shape=(32,32,3))
x = data_augmentation(inputs_aug)
x = layers.Conv2D(32,3,padding="same",activation="relu")(x)
x = layers.Conv2D(32,3,padding="same",activation="relu")(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(64,3,padding="same",activation="relu")(x)
x = layers.Conv2D(64,3,padding="same",activation="relu")(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(128,3,padding="same",activation="relu")(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256,activation="relu")(x)
outputs_aug = layers.Dense(10,activation="softmax")(x)
model_keras_aug = keras.Model(inputs_aug,outputs_aug,name="cnn_keras_aug")
model_keras_aug.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])
model_keras_aug.summary()

In [ ]:
# Train Keras augmentation model
es_keras = keras.callbacks.EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True,verbose=1)
rlrop_keras = keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=5,min_lr=1e-6,verbose=1)
history_keras_aug = model_keras_aug.fit(X_train_aug,y_train_aug,epochs=100,batch_size=64,
    validation_data=(X_val_aug,y_val_aug),callbacks=[es_keras,rlrop_keras],verbose=1)
test_loss_keras_aug,test_accuracy_keras_aug = model_keras_aug.evaluate(X_test,y_test_ohe,verbose=0)
print("Keras augmentation test accuracy:",test_accuracy_keras_aug)

In [ ]:
# Compare validation accuracy of the two augmentation approaches
plt.figure(figsize=(8,4))
plt.plot(history_aug.history["val_accuracy"],label="ImageDataGenerator")
plt.plot(history_keras_aug.history["val_accuracy"],label="Keras augmentation layers")
plt.title("Augmentation Validation Accuracy Comparison"); plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.legend(); plt.tight_layout(); plt.show()

## A6 — MobileNetV2 transfer learning, results table, residual connection

In [ ]:
# Resize batches to 96x96 on demand to avoid storing all resized images in RAM
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE_TL = 64
X_tl_train, y_tl_train = X_train[:-validation_size], y_train_ohe[:-validation_size]
X_tl_val, y_tl_val = X_train[-validation_size:], y_train_ohe[-validation_size:]

def make_resized_dataset(images, labels, batch_size=64, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((images,labels))
    if shuffle:
        ds = ds.shuffle(10000,seed=42)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda image,label:(tf.image.resize(image,(96,96)),label),
                num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_tl_ds = make_resized_dataset(X_tl_train,y_tl_train,BATCH_SIZE_TL,True)
val_tl_ds = make_resized_dataset(X_tl_val,y_tl_val,BATCH_SIZE_TL)
test_tl_ds = make_resized_dataset(X_test,y_test_ohe,BATCH_SIZE_TL)

In [ ]:
# Load pretrained MobileNetV2 without its original classifier
base_model = keras.applications.MobileNetV2(input_shape=(96,96,3),include_top=False,weights="imagenet")
base_model.trainable = False

In [ ]:
# Map normalized pixels [0,1] to [-1,1], expected by MobileNetV2
inputs_tl = keras.Input(shape=(96,96,3))
x = layers.Rescaling(scale=2.0,offset=-1.0)(inputs_tl)
x = base_model(x,training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128,activation="relu")(x)
outputs_tl = layers.Dense(10,activation="softmax")(x)
model_transfer = keras.Model(inputs_tl,outputs_tl,name="mobilenetv2_transfer")
model_fe = model_transfer
model_fe.compile(optimizer=keras.optimizers.Adam(0.001),loss="categorical_crossentropy",metrics=["accuracy"])
es_fe = keras.callbacks.EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True,verbose=1)
rlrop_fe = keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=3,min_lr=1e-6,verbose=1)

In [ ]:
# Feature extraction: only the new classification head is trainable
history_fe = model_fe.fit(train_tl_ds,epochs=20,validation_data=val_tl_ds,callbacks=[es_fe,rlrop_fe])
test_loss_fe,test_accuracy_fe = model_fe.evaluate(test_tl_ds,verbose=0)
print("MobileNetV2 feature extraction accuracy:",test_accuracy_fe)

In [ ]:
# Fine-tune only the final 30 base-model layers; keep BatchNorm frozen
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer,layers.BatchNormalization):
        layer.trainable = False

model_fe.compile(optimizer=keras.optimizers.Adam(1e-5),loss="categorical_crossentropy",metrics=["accuracy"])
es_ft = keras.callbacks.EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True,verbose=1)
rlrop_ft = keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=3,min_lr=1e-7,verbose=1)
history_ft = model_fe.fit(train_tl_ds,epochs=10,validation_data=val_tl_ds,callbacks=[es_ft,rlrop_ft])
test_loss_ft,test_accuracy_ft = model_fe.evaluate(test_tl_ds,verbose=0)
print("MobileNetV2 fine-tuned accuracy:",test_accuracy_ft)

In [ ]:
# Plot feature-extraction and fine-tuning validation accuracy
plt.figure(figsize=(8,4))
plt.plot(range(1,len(history_fe.history["val_accuracy"])+1),history_fe.history["val_accuracy"],label="Feature extraction")
plt.plot(range(len(history_fe.history["val_accuracy"])+1,len(history_fe.history["val_accuracy"])+len(history_ft.history["val_accuracy"])+1),
         history_ft.history["val_accuracy"],label="Fine-tuning")
plt.axvline(len(history_fe.history["val_accuracy"])+0.5,linestyle="--",label="Fine-tuning starts")
plt.title("MobileNetV2 Validation Accuracy"); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()
plt.tight_layout(); plt.show()

### A6.5 — Full results comparison table
The next cell compares all six required models and highlights the highest measured test accuracy.

In [ ]:
# Create the required comparison table using measured test results
results_df = pd.DataFrame([
    {"Model":"LeNet-5 (A3.3)","Architecture":"LeNet-5 style CNN","Augmentation":"No","Transfer Learning":"No","Test Accuracy":test_accuracy_lenet,"Parameters":lenet5.count_params()},
    {"Model":"CNN Scratch No Aug (A4)","Architecture":"Custom CNN","Augmentation":"No","Transfer Learning":"No","Test Accuracy":test_accuracy_cnn,"Parameters":model_cnn.count_params()},
    {"Model":"CNN Scratch + Aug (A5.3)","Architecture":"Custom CNN","Augmentation":"ImageDataGenerator","Transfer Learning":"No","Test Accuracy":test_accuracy_aug,"Parameters":model_aug.count_params()},
    {"Model":"CNN + Keras Aug Layers (A5.4)","Architecture":"Custom CNN","Augmentation":"Keras preprocessing layers","Transfer Learning":"No","Test Accuracy":test_accuracy_keras_aug,"Parameters":model_keras_aug.count_params()},
    {"Model":"MobileNetV2 Feature Extraction (A6.3)","Architecture":"MobileNetV2","Augmentation":"No","Transfer Learning":"Frozen pretrained base","Test Accuracy":test_accuracy_fe,"Parameters":model_transfer.count_params()},
    {"Model":"MobileNetV2 Fine-Tuned (A6.4)","Architecture":"MobileNetV2","Augmentation":"No","Transfer Learning":"Last 30 layers fine-tuned","Test Accuracy":test_accuracy_ft,"Parameters":model_fe.count_params()}
]).sort_values("Test Accuracy",ascending=False).reset_index(drop=True)

In [ ]:
# Highlight the maximum test accuracy
display(results_df.style.highlight_max(subset=["Test Accuracy"],color="lightgreen")
        .format({"Test Accuracy":"{:.2%}","Parameters":"{:,}"}))

**Accuracy progression and effort–accuracy trade-off:** Use the measured table above to state which model performed best. LeNet-5 is a compact baseline; a custom CNN learns features from CIFAR-10 directly. Augmentation adds variation and may improve generalization, but increases training work. MobileNetV2 feature extraction reuses pretrained features with a frozen base and often reduces training effort. Fine-tuning takes more computation but can adapt pretrained features to CIFAR-10. Improvements are not guaranteed; describe the actual results rather than assuming a fixed progression.

### A6.6 — Non-linear architecture: residual connection

In [ ]:
# Create a Functional API input
residual_inputs = keras.Input(shape=(32,32,3),name="residual_input")

In [ ]:
# Shortcut path projects RGB input to 64 channels using a 1x1 convolution
shortcut = layers.Conv2D(64,(1,1),padding="same",name="shortcut_conv")(residual_inputs)

In [ ]:
# Main path contains two 3x3 convolutions with ReLU
main = layers.Conv2D(64,(3,3),padding="same",activation="relu",name="main_conv1")(residual_inputs)
main = layers.Conv2D(64,(3,3),padding="same",activation="relu",name="main_conv2")(main)

In [ ]:
# Add both paths and apply final ReLU
added = layers.Add(name="residual_add")([shortcut,main])
residual_outputs = layers.Activation("relu",name="residual_output_relu")(added)

In [ ]:
# Wrap the residual block in a Functional API model
residual_model = keras.Model(residual_inputs,residual_outputs,name="minimal_residual_block")
residual_model.summary()

In [ ]:
# Verify output shape with one sample
sample_image = tf.random.normal((1,32,32,3))
sample_output = residual_model(sample_image)
print("Input shape:",sample_image.shape)
print("Output shape:",sample_output.shape)

**Residual connections:** Non-sequential architectures allow skip connections, giving gradients a path that bypasses some layers during backpropagation. This helps mitigate vanishing gradients and supports training very deep networks, although it does not guarantee that the problem is completely eliminated. Residual learning is the core innovation of ResNet (He et al., 2016).

Reference: He, K., Zhang, X., Ren, S., & Sun, J. (2016). *Deep Residual Learning for Image Recognition*. CVPR.

## Conclusion and submission checklist
This project compares CNNs trained from scratch, data augmentation, and pretrained MobileNetV2 transfer learning. The comparison table uses measured test accuracy and parameter counts.

- [ ] Restart the kernel and run every cell from top to bottom.
- [ ] Confirm no errors and all six models have test accuracy values.
- [ ] Review plots, confusion matrix, and the final results table.
- [ ] Save as `DL_PR4A.ipynb`.
- [ ] Export to HTML if required by the instructor.
- [ ] Submit `README.md` and `requirements.txt` with the notebook.